In [ ]:
pip install gdown

In [ ]:
!gdown --continue https://drive.google.com/uc?id=1lhAaeQCmk2y440PmagA0KmIVBIysVMwu -O tennis_court_det_dataset.zip


In [ ]:
!mkdir -p dataset
!unzip tennis_court_det_dataset.zip -d dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models


import albumentations as A
from albumentations.pytorch import ToTensorV2

import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt



In [ ]:


class CourtKeypointsDataset(Dataset):
    def __init__(
        self,
        annotations_file="/content/dataset/data/data_train.json",
        images_path="/content/dataset/data/images",
        read_from_stubs=False,
        save_path="/content/dataset/annotations.json",
        img_size = 224
    ):
        self.annotations_file = annotations_file
        self.images_path = images_path
        self.save_path = save_path
        self.img_size = img_size

        # ✅ Keypoint-aware transforms
        self.transforms = A.Compose(
            [
                A.Resize(self.img_size, self.img_size),
                A.RandomBrightnessContrast(p=0.2),
                A.Normalize(
                    mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)
                ),
                ToTensorV2()
            ],
            keypoint_params=A.KeypointParams(format="xy", remove_invisible=False)
        )

        if read_from_stubs and os.path.exists(self.save_path):
            self.load_annotations_dict()
        else:
            self.load_annotations_file()
            self.create_annotations_dict()
            self.save_annotations_dict()

    # --------------------------------------------------
    # Load raw JSON
    # --------------------------------------------------
    def load_annotations_file(self):
        with open(self.annotations_file, "r") as f:
            self.data = json.load(f)

    # --------------------------------------------------
    # Create internal annotation structure
    # --------------------------------------------------
    def create_annotations_dict(self):
        """
        Expected JSON format:
        [
          {
            "id": 0,
            "file_name": "0001.jpg",
            "kps": [[x,y], [x,y], ...]
          }
        ]
        """
        self.annotations = []

        for item in self.data:
            image_id = item["id"]
            keypoints = item["kps"]  # [(x,y), ...]

            self.annotations.append({
                "id": image_id,
                "keypoints": keypoints
            })

    # --------------------------------------------------
    # Save processed annotations
    # --------------------------------------------------
    def save_annotations_dict(self):
        with open(self.save_path, "w") as f:
            json.dump(self.annotations, f, indent=2)

    # --------------------------------------------------
    # Load processed annotations
    # --------------------------------------------------
    def load_annotations_dict(self):
        with open(self.save_path, "r") as f:
            self.annotations = json.load(f)

    # --------------------------------------------------
    # Dataset interface
    # --------------------------------------------------
    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]

        # Use the correct filename
        img_path = os.path.join(self.images_path, f"{ann['id']}.png")
        image = cv2.imread(img_path)

        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        keypoints = ann["keypoints"]  # [(x,y), ...]

        # Apply Albumentations transforms
        augmented = self.transforms(image=image, keypoints=keypoints)
        image = augmented["image"]
        keypoints = np.array(augmented["keypoints"], dtype=np.float32)

        # normalize keypoints to [0, 1]
        keypoints[:, 0] /= self.img_size
        keypoints[:, 1] /= self.img_size

        # ✅ Flatten keypoints
        flat_keypoints = torch.tensor(keypoints, dtype=torch.float32).view(-1)


        return image, flat_keypoints



In [ ]:

# Get image and keypoints
a = CourtKeypointsDataset()
image, flat_keypoints = a.__getitem__(5)

# If flat_keypoints is [x1, y1, x2, y2, ...], convert to [(x,y), ...]
img_size = 224

keypoints = [
    (flat_keypoints[i] * img_size, flat_keypoints[i+1] * img_size)
    for i in range(0, len(flat_keypoints), 2)
]

# Plot image
plt.figure(figsize=(6,6))
plt.imshow(image.permute(1, 2, 0))  # convert C,H,W → H,W,C if it's a tensor
plt.axis('off')

# Overlay keypoints
xs, ys = zip(*keypoints)
plt.scatter(xs, ys, c='red', s=30)

plt.show()


In [ ]:
class KeypointsModel(nn.Module):
    def __init__(self, num_keypoints=14):
        super(KeypointsModel, self).__init__()

        # Load a pretrained ResNet backbone (e.g., resnet18)
        self.backbone = models.resnet18(pretrained=True)

        # Remove the final classification layer
        # ResNet has: fc = Linear(512, 1000)
        self.backbone.fc = nn.Identity()  # output: feature vector of size 512

        # Fully connected layer to predict all keypoints
        self.fc = nn.Linear(512, num_keypoints * 2)

    def forward(self, images):
        """
        images: Tensor of shape (B, 3, H, W)
        returns: Tensor of shape (B, num_keypoints*2)
        """
        features = self.backbone(images)  # shape: (B, 512)
        out = self.fc(features)           # shape: (B, num_keypoints*2)
        return out


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

def main():
    # ------------------------------
    # 1. Hyperparameters
    # ------------------------------
    annotations_file = "/content/dataset/data/data_train.json"
    images_path = "/content/dataset/data/images"
    num_keypoints = 14
    batch_size = 8
    lr = 1e-3
    num_epochs = 10
    val_split = 0.1  # fraction for validation

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    # ------------------------------
    # 2. Dataset & DataLoader
    # ------------------------------
    dataset = CourtKeypointsDataset(
        annotations_file=annotations_file,
        images_path=images_path
    )

    # Train / validation split
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # ------------------------------
    # 3. Model, Loss, Optimizer
    # ------------------------------
    model = KeypointsModel(num_keypoints=num_keypoints).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # ------------------------------
    # 4. Training loop
    # ------------------------------
    for epoch in range(num_epochs):
        print(f"STARTING OF THE EPOCH {epoch}")
        model.train()
        running_loss = 0.0

        for images, keypoints in train_loader:
            images = images.to(device)                     # (B, 3, H, W)
            keypoints = keypoints.to(device).float()      # (B, num_keypoints*2)

            optimizer.zero_grad()
            outputs = model(images)                       # (B, num_keypoints*2)
            print(outputs.shape, keypoints.shape)
            loss = criterion(outputs, keypoints)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        # ------------------------------
        # Validation
        # ------------------------------
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, keypoints in val_loader:
                images = images.to(device)
                keypoints = keypoints.to(device).float()
                outputs = model(images)
                loss = criterion(outputs, keypoints)
                val_loss += loss.item() * images.size(0)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_loss:.4f} "
              f"Val Loss: {val_loss:.4f}")

    print("Training finished.")

    # ------------------------------
    # 5. Save model
    # ------------------------------
    torch.save(model.state_dict(), "keypoints_model.pth")
    print("Model saved as keypoints_model.pth")

if __name__ == "__main__":
    main()


In [ ]:
# # Loading the model

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = KeypointsModel(num_keypoints=14)
# model.load_state_dict(torch.load("court_keypoints_model.pth", map_location=device))
# model.to(device)
# model.eval()